# MCP Intro: Why Discovery Matters

This notebook explains the core idea behind the Model Context Protocol (MCP) and why discovery, metadata, and clean interface definitions matter for modern AI systems.

## What this notebook shows

1. Why hardcoded tool wiring becomes fragile as toolsets grow.
2. How metadata enables discovery and safer invocation.
3. What MCP adds on top of a metadata-driven registry.
4. How this conceptual model leads into the real MCP demo in the next notebook.

## Problem: hardcoded application wiring

When an application calls services directly, the code is tightly coupled to specific function names, parameter orders, and usage patterns. That works for one or two tools, but once you add more capabilities it becomes hard to maintain, understand, and extend.

In [ ]:
from typing import Any, Dict, List
import json

CUSTOMERS = {
    "C001": {
        "name": "Ava Chen",
        "email": "ava.chen@example.com",
        "segment": "student",
    },
    "C002": {
        "name": "Marcus Rivera",
        "email": "marcus.rivera@example.com",
        "segment": "creator",
    },
}

INVENTORY = [
    {"sku": "LAP-101", "name": "FeatherBook 13", "category": "laptops", "price": 899},
    {"sku": "MON-310", "name": "ColorView 27", "category": "monitors", "price": 399},
]

def lookup_customer(customer_id: str) -> Dict[str, Any]:
    customer = CUSTOMERS.get(customer_id)
    if not customer:
        return {"ok": False, "error": f"No customer found for id {customer_id}"}
    return {"ok": True, "customer": customer}

def lookup_inventory(category: str | None = None) -> Dict[str, Any]:
    items = [item for item in INVENTORY if category is None or item["category"] == category]
    return {"ok": True, "items": items}

def print_json(value: Any) -> None:
    print(json.dumps(value, indent=2))

print_json(lookup_customer("C001"))
print_json(lookup_inventory("laptops"))


## What is brittle about this approach?

- Each caller must know exact function names and arguments.
- It is hard to discover new capabilities automatically.
- The system cannot easily route requests to the right tool on its own.

A better approach is to describe capabilities with metadata, then discover and invoke them dynamically.

## Part 2: MCP-shaped discovery

Instead of hardcoding each call, we can describe tools with metadata and use a registry to discover them. This is the core idea behind MCP: a machine-readable contract layer for tools and resources.

In [ ]:
TOOL_REGISTRY = {
    "lookup_customer": {
        "description": "Find customer details by customer ID.",
        "parameters": ["customer_id"],
    },
    "lookup_inventory": {
        "description": "List inventory for a category.",
        "parameters": ["category"],
    },
}

def invoke_tool(tool_name: str, **kwargs: Any) -> Dict[str, Any]:
    if tool_name == "lookup_customer":
        return lookup_customer(**kwargs)
    if tool_name == "lookup_inventory":
        return lookup_inventory(**kwargs)
    return {"ok": False, "error": "Unknown tool"}

print_json(TOOL_REGISTRY)
print_json(invoke_tool("lookup_inventory", category="laptops"))


## Takeaways

- Hardcoded wiring is simple, but it does not scale.
- Metadata-driven discovery makes the system easier to extend and safer to automate.
- MCP builds on this idea by standardizing how clients discover servers, resources, and tools.

In the next notebook, we will move from this conceptual model to a real MCP client/server demo with a constrained, agentic workflow.